In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

# For reproducibility (optional)
torch.manual_seed(0)
np.random.seed(0)

# Hyperparams
batch_size = 64
num_workers = 2

In [2]:
# MNIST transforms (normalize to [0,1], optionally we could standardize further)
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
)

print("Number of training samples:", len(train_dataset))
print("Number of test samples:", len(test_dataset))

Failed to download (trying next):
<urlopen error [Errno 111] Connection refused>



100%|██████████| 9.91M/9.91M [00:00<00:00, 68.5MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
<urlopen error [Errno 111] Connection refused>



100%|██████████| 28.9k/28.9k [00:00<00:00, 2.51MB/s]

Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
<urlopen error [Errno 111] Connection refused>




100%|██████████| 1.65M/1.65M [00:00<00:00, 19.1MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
<urlopen error [Errno 111] Connection refused>



100%|██████████| 4.54k/4.54k [00:00<00:00, 24.6MB/s]


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

Number of training samples: 60000
Number of test samples: 10000


In [3]:
def pairwise_correlation_penalty(h):
    """
    h: tensor of shape (batch_size, num_neurons)
    Returns a scalar penalty measuring the average pairwise correlation in the batch.
    """
    # Center each neuron's activation (subtract mean per neuron).
    h_centered = h - h.mean(dim=0, keepdim=True)  # shape: (B, D)
    
    # Compute covariance matrix among neurons: shape (D, D)
    # Cov(X) = (1/(N-1)) * X^T X, but we just do the raw dot product for approximation
    cov = torch.matmul(h_centered.t(), h_centered)
    
    # Variances on the diagonal
    diag = torch.diagonal(cov)
    # Expand the variance terms to form stdevs
    stddev = torch.sqrt(diag + 1e-6)  # add small epsilon to avoid /0
    
    # Correlation matrix R[i,j] = Cov[i,j] / (std[i]*std[j])
    outer_stddev = torch.ger(stddev, stddev)  # shape (D, D)
    corr_matrix = cov / (outer_stddev + 1e-6)
    
    # We only want upper triangular off-diagonal elements
    D = h.shape[1]
    corr_off_diag = []
    for i in range(D):
        for j in range(i+1, D):
            corr_off_diag.append(corr_matrix[i,j])
    
    # Convert to tensor
    corr_off_diag = torch.stack(corr_off_diag)
    
    # Return the mean absolute correlation as the penalty
    penalty = torch.mean(torch.abs(corr_off_diag))
    return penalty

In [4]:
class SingleLayerBlock(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes=10):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.linear1 = nn.Linear(in_dim, hidden_dim)
        self.relu = nn.ReLU()
        # Classification head from hidden_dim -> num_classes
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        """
        x: shape (batch_size, in_dim)
        Returns:
          logits: shape (batch_size, num_classes)
          hidden: shape (batch_size, hidden_dim)
        """
        h = self.linear1(x)
        h = self.relu(h)
        logits = self.classifier(h)
        return logits, h

In [5]:
def train_single_layer_block(
    model, 
    train_loader, 
    lr=1e-3, 
    weight_decay=1e-5, 
    corr_lambda=0.01,
    epochs=5,
    device='cpu'
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    model.to(device)
    model.train()
    
    for epoch in range(epochs):
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)  # flatten
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            logits, h = model(images)  # forward
            ce_loss = criterion(logits, labels)
            
            # correlation penalty among hidden units
            corr_pen = pairwise_correlation_penalty(h)
            
            loss = ce_loss + corr_lambda * corr_pen
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f}")
    return model

In [6]:
def evaluate_accuracy(model, data_loader, device='cpu'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            logits, _ = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

In [7]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
first_layer_model = SingleLayerBlock(in_dim=28*28, hidden_dim=128).to(device)

first_layer_model = train_single_layer_block(
    model=first_layer_model,
    train_loader=train_loader,
    lr=1e-3,
    weight_decay=1e-5,
    corr_lambda=0.01,
    epochs=5,
    device=device
)
acc_train = evaluate_accuracy(first_layer_model, train_loader, device)
acc_test = evaluate_accuracy(first_layer_model, test_loader, device)
print(f"Train Accuracy (1-layer): {acc_train:.2f}%")
print(f"Test Accuracy  (1-layer): {acc_test:.2f}%")

Epoch [1/5] - Loss: 0.3511
Epoch [2/5] - Loss: 0.1685
Epoch [3/5] - Loss: 0.1174
Epoch [4/5] - Loss: 0.0890
Epoch [5/5] - Loss: 0.0703
Train Accuracy (1-layer): 98.26%
Test Accuracy  (1-layer): 97.29%


In [8]:
# Freeze parameters of first layer
for param in first_layer_model.parameters():
    param.requires_grad = False

# We'll create new data loaders that hold the hidden representation as input
X_train_hidden = []
y_train_list = []
first_layer_model.eval()

with torch.no_grad():
    for images, labels in train_loader:
        images = images.view(images.size(0), -1).to(device)
        _, h = first_layer_model(images)
        X_train_hidden.append(h.cpu())
        y_train_list.append(labels)

X_train_hidden = torch.cat(X_train_hidden, dim=0)
y_train = torch.cat(y_train_list, dim=0)

# Similarly for test set
X_test_hidden = []
y_test_list = []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(images.size(0), -1).to(device)
        _, h = first_layer_model(images)
        X_test_hidden.append(h.cpu())
        y_test_list.append(labels)

X_test_hidden = torch.cat(X_test_hidden, dim=0)
y_test = torch.cat(y_test_list, dim=0)

print("Shape of X_train_hidden:", X_train_hidden.shape)
print("Shape of X_test_hidden:", X_test_hidden.shape)

Shape of X_train_hidden: torch.Size([60000, 128])
Shape of X_test_hidden: torch.Size([10000, 128])


In [9]:
train_hidden_dataset = torch.utils.data.TensorDataset(X_train_hidden, y_train)
test_hidden_dataset = torch.utils.data.TensorDataset(X_test_hidden, y_test)

train_hidden_loader = torch.utils.data.DataLoader(
    train_hidden_dataset, batch_size=batch_size, shuffle=True
)
test_hidden_loader = torch.utils.data.DataLoader(
    test_hidden_dataset, batch_size=batch_size, shuffle=False
)

In [10]:
second_layer_model = SingleLayerBlock(in_dim=128, hidden_dim=64, num_classes=10).to(device)

second_layer_model = train_single_layer_block(
    model=second_layer_model,
    train_loader=train_hidden_loader,
    lr=1e-3,
    weight_decay=1e-5,
    corr_lambda=0.01,
    epochs=5,
    device=device
)

# Evaluate on hidden test data
acc_train2 = evaluate_accuracy(second_layer_model, train_hidden_loader, device)
acc_test2 = evaluate_accuracy(second_layer_model, test_hidden_loader, device)

print(f"Train Accuracy (2nd layer): {acc_train2:.2f}%")
print(f"Test Accuracy  (2nd layer): {acc_test2:.2f}%")

Epoch [1/5] - Loss: 0.1800
Epoch [2/5] - Loss: 0.0585
Epoch [3/5] - Loss: 0.0497
Epoch [4/5] - Loss: 0.0461
Epoch [5/5] - Loss: 0.0432
Train Accuracy (2nd layer): 98.88%
Test Accuracy  (2nd layer): 97.73%
